In [ ]:
! pip install transformers torch accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_PATH = r"C:\qwen1.5b"
SYSTEM_PROMPT = "You are a helpful technical assistant. Answer clearly and accurately."
MAX_TOKENS = 512

print("Loading model... (1-2 minutes)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,   # CPU
    device_map="cpu",
)
model.eval()

print("=" * 40)
print("   Qwen2.5-1.5B  |  Offline  |  CPU")
print("   'clear' = new chat  |  'quit' = exit")
print("=" * 40 + "\n")

history = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(user_input):
    history.append({"role": "user", "content": user_input})

    prompt = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": response})
    return response


while True:
    try:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "bye"):
            print("Bye!")
            break
        if user_input.lower() == "clear":
            history.clear()
            history.append({"role": "system", "content": SYSTEM_PROMPT})
            print("--- Chat cleared ---\n")
            continue
        print("Thinking...\n")
        reply = chat(user_input)
        print(f"Bot: {reply}\n")
        print("-" * 40)
    except KeyboardInterrupt:
        print("\nBye!")
        break

In [ ]:
# version 2
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = r"C:\qwen1.5b"

SYSTEM_PROMPT = """
You are an offline exam MCQ assistant.

Return:
Answer: <option letter>
Reason: <one short sentence>

Do not repeat the question.
Do not repeat all options.
Keep the response under 50 words.
"""

MAX_TOKENS = 64

# ============================================================
# LOAD MODEL
# ============================================================

print("Loading model... Please wait.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="cpu",
    local_files_only=True
)

model.eval()

print("=" * 50)
print("        OFFLINE EXAM MCQ ASSISTANT")
print("        Model : Local Qwen")
print("        Device: CPU")
print("        Precision: FP16")
print("        Max tokens: 64")
print("=" * 50)
print()
print("Type 'clear' to clear chat.")
print("Type 'quit' to exit.")
print()


# ============================================================
# CHAT FUNCTION
# ============================================================

def answer_mcq(question):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # Create prompt using the model's chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # CPU inference
    with torch.no_grad():

        output = model.generate(
            **inputs,

            max_new_tokens=MAX_TOKENS,

            # Deterministic answer
            do_sample=False,

            # Avoid excessive repetition
            repetition_penalty=1.05,

            # Stop at EOS
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    # Get only newly generated tokens
    input_length = inputs["input_ids"].shape[1]

    new_tokens = output[0][input_length:]

    # Decode answer
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return response


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    try:

        question = input("You: ").strip()

        if not question:
            continue

        # Exit
        if question.lower() in ["quit", "exit", "bye"]:
            print("\nBye!")
            break

        # Clear
        if question.lower() == "clear":
            print("\n--- Ready for next question ---\n")
            continue

        print("\nThinking...\n")

        answer = answer_mcq(question)

        print("Bot:")
        print(answer)

        print("\n" + "-" * 50 + "\n")

    except KeyboardInterrupt:

        print("\n\nBye!")
        break

    except Exception as e:

        print("\nError:", e)
        print()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

# ============================================================
# CONFIG
# ============================================================

MODEL_PATH = r"/home/souvik/Desktop/AI-for-exam-main/qwen1.5b"  # Linux path

SYSTEM_PROMPT = "You are an MCQ solver. Reply with only A, B, C, or D. No explanation."

# ============================================================
# LOAD
# ============================================================

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.float32,          # fixed deprecation warning
    device_map="cpu",
    local_files_only=True,
    low_cpu_mem_usage=True,
)
model.eval()

print()
print("=" * 40)
print("   OFFLINE MCQ SOLVER  |  CPU  |  FP32")
print("   Paste question, then press Enter twice")
print("   Type 'quit' to exit")
print("=" * 40)
print()

# ============================================================
# SOLVE FUNCTION
# ============================================================

def solve(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_token = output[0][input_len:]
    answer = tokenizer.decode(new_token, skip_special_tokens=True).strip().upper()

    for opt in ("A", "B", "C", "D"):
        if opt in answer:
            return opt

    return "?"

# ============================================================
# MULTILINE INPUT FUNCTION
# ============================================================

def get_multiline_input():
    """
    Keeps reading lines until user presses Enter on an empty line.
    That empty line = end of question.
    """
    print("MCQ (press Enter twice when done):")
    lines = []
    while True:
        line = input()
        if line.strip().lower() in ("quit", "exit", "bye"):
            return "quit"
        if line == "":          # blank line = done
            if lines:           # only stop if we have something
                break
        else:
            lines.append(line)
    return "\n".join(lines)

# ============================================================
# MAIN LOOP
# ============================================================

while True:
    try:
        question = get_multiline_input()

        if not question:
            continue

        if question == "quit":
            print("Bye!")
            break

        print("Thinking...")
        start = time.perf_counter()
        answer = solve(question)
        elapsed = time.perf_counter() - start

        print(f"ANSWER : {answer}")
        print(f"Time   : {elapsed:.2f} sec")
        print()

    except KeyboardInterrupt:
        print("\nBye!")
        break

    except Exception as e:
        print("ERROR:", e)
        print()

## version 4

In [ ]:
pip install llama-cpp-python

In [ ]:
# https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF
# qwen2.5-7b-instruct-q4_k_m.gguf     (~5 GB)

In [ ]:
from llama_cpp import Llama
import time

# ============================================================
# CONFIG
# ============================================================

MODEL_PATH = "/home/souvik/Desktop/AI-for-exam-main/qwen7b/qwen2.5-7b-instruct-q4_k_m.gguf"

SYSTEM_PROMPT = """You are a Java programming expert.
Analyze the Java MCQ carefully. Think about Java syntax, rules, and behavior.
Reply with ONLY one letter: A, B, C, or D. Nothing else."""

# ============================================================
# LOAD MODEL
# ============================================================

print("Loading model...")

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,          # context window
    n_threads=4,         # CPU threads — adjust to your CPU cores
    n_gpu_layers=0,      # 0 = full CPU
    verbose=False,
)

print()
print("=" * 40)
print("   JAVA MCQ SOLVER  |  7B  |  ACCURATE")
print("   Paste question, press Enter twice")
print("   Type 'quit' to exit")
print("=" * 40)
print()

# ============================================================
# SOLVE FUNCTION
# ============================================================

def solve(question):
    prompt = f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{question}

Correct answer (A/B/C/D):<|im_end|>
<|im_start|>assistant
"""

    response = llm(
        prompt,
        max_tokens=8,
        temperature=0.0,      # deterministic
        stop=["<|im_end|>", "\n\n"],
        echo=False,
    )

    raw = response["choices"][0]["text"].strip().upper()

    for ch in raw:
        if ch in ("A", "B", "C", "D"):
            return ch

    return "?"

# ============================================================
# MULTILINE INPUT
# ============================================================

def get_multiline_input():
    print("MCQ (press Enter twice when done):")
    lines = []
    while True:
        line = input()
        if line.strip().lower() in ("quit", "exit", "bye"):
            return "quit"
        if line == "":
            if lines:
                break
        else:
            lines.append(line)
    return "\n".join(lines)

# ============================================================
# MAIN LOOP
# ============================================================

while True:
    try:
        question = get_multiline_input()

        if not question:
            continue

        if question == "quit":
            print("Bye!")
            break

        print("Thinking...")
        start = time.perf_counter()
        answer = solve(question)
        elapsed = time.perf_counter() - start

        print(f"\nANSWER : {answer}")
        print(f"Time   : {elapsed:.2f} sec\n")

    except KeyboardInterrupt:
        print("\nBye!")
        break

    except Exception as e:
        print("ERROR:", e)
        print()